In [7]:
import numpy as np
import astropy.io.ascii
import pandas as pd

This notebook looks at the data for the sources.

In [8]:
targets = [
    "HH270VLA1",
    "HOPS-323",
    "HOPS-312",
    "HOPS-364",
    "HOPS-45",
    "HOPS-361", # file is named 361-N
    "HOPS-366", 
    "HOPS-384",
    "HOPS-304",
    "HOPS-357",
    "HOPS-363",
    "HOPS-400",
    "HOPS-290",
    "HOPS-56",
    "HOPS-193",
    "HOPS-242",
    "HOPS-70",
    "HOPS-138",
    "HOPS-85",
    "HOPS-182",
    "HOPS-28",
    "HOPS-92",
    "HOPS-32",
    "HOPS-158",
    "HOPS-75",
    "HOPS-395",
    "HOPS-43",
    "HOPS-255",
    "HOPS-213",
    "HOPS-77",
    "HOPS-281",
    "HOPS-173",
    "HOPS-282",
    "HOPS-84",
    "HOPS-288",
    "HOPS-248",
    "HOPS-168",
    "HOPS-203",
    "HOPS-163",
    "HOPS-12",
    "HOPS-261"
]

In [9]:
source_info = astropy.io.ascii.read("../data/distances.txt").to_pandas()
# filter for relevant targets
source_info = source_info[source_info['Main'].isin(targets)]

# create ra/dec columns
source_info['RA'] = 15 * (source_info['RAh'] + source_info['RAm'] / 60 + source_info['RAs'] / 3600)
source_info['DE-'] = source_info["DE-"].apply(lambda x: -1 if x == "-" else 1)
source_info['Dec'] = source_info['DE-'] * (source_info['DEd'] + source_info['DEm'] / 60 + source_info['DEs'] / 3600)

# order columns and sort
source_info = source_info[['Main', 'Source', 'RA', 'Dec', 'Dis', 'LBol', 'TBol', 'Class', 'SigmaYSO']]
source_info = source_info.sort_values(['Main', 'Source']).reset_index(drop=True)

# fix HOPS-361-N
source_info["Main"] = source_info["Main"].replace({"HOPS-361": "HOPS-361-N"})
source_info["Source"] = source_info["Source"].apply(lambda x: str(x).replace("HOPS-361", "HOPS-361-N"))
source_info[source_info["Main"] == "HOPS-361-N"]

# save
source_info.to_csv("../data/source_info.csv",index=False)
source_info

,Main,Source,RA,Dec,Dis,LBol,TBol,Class,SigmaYSO
0,HH270VLA1,HH270VLA1-A,87.894113,2.946114,430.1,7.34,32.0,0,2.2
1,HH270VLA1,HH270VLA1-B,87.894167,2.946078,430.1,7.34,32.0,0,2.2
2,HOPS-12,HOPS-12-A,83.785962,-5.931844,388.6,6.26,42.0,0,49.8
3,HOPS-12,HOPS-12-B-A,83.787271,-5.931953,388.6,6.26,42.0,0,49.8
4,HOPS-12,HOPS-12-B-B,83.787312,-5.931911,388.6,6.26,42.0,0,49.8
...,...,...,...,...,...,...,...,...,...
106,HOPS-85,HOPS-85-A,83.867454,-5.061444,392.8,14.24,174.2,Flat,77.4
107,HOPS-85,HOPS-85-B,83.867471,-5.061469,392.8,14.24,174.2,Flat,77.4
108,HOPS-92,HOPS-92-A-A,83.826404,-5.009150,392.7,17.58,186.3,Flat,139.2
109,HOPS-92,HOPS-92-A-B,83.826367,-5.009217,392.7,17.58,186.3,Flat,139.2


In [10]:
def split_source_string(x):
    x = str(x).split('+')
    if len(x[0]) == 1:
        a = x[0]
    else:
        a = x[0][0] + '-' + x[0][1]
    if len(x[1]) == 1:
        b = x[1]
    else:
        b = x[1][0] + '-' + x[1][1]
    return [a, b]

# measurements = pd.read_csv("../data/measurements.csv")
# measurements = measurements[['Field', 'Outflow Source', 'Average Angle (Blue)']]
# measurements['source_a'] = measurements['Outflow Source'].apply(lambda x: split_source_string(x)[0])
# measurements['source_b'] = measurements['Outflow Source'].apply(lambda x: split_source_string(x)[1])
# measurements = measurements[['Field', 'source_a', 'source_b', 'Average Angle (Blue)']].rename(columns={'Field': 'field', 'Average Angle (Blue)': 'angle'})

# new_rows = []
# for i, row in measurements.iterrows():
#     key_a = row['field'] + '-' + row['source_a']
#     key_b = row['field'] + '-' + row['source_b']
#     row_a = source_info.loc[source_info['Source'] == key_a]
#     row_b = source_info.loc[source_info['Source'] == key_b]
#     row['source_a_ra'] = row_a['RA'].iloc[0]
#     row['source_a_dec'] = row_a['Dec'].iloc[0]
#     row['source_b_ra'] = row_b['RA'].iloc[0]
#     row['source_b_dec'] = row_b['Dec'].iloc[0]
#     row['distance'] = row_a['Dis'].iloc[0]
#     new_rows.append(row)

# master = pd.DataFrame(new_rows)[['field', 'source_a', 'source_a_ra', 'source_a_dec', 'source_b', 'source_b_ra', 'source_b_dec', 'distance', 'angle']]
# master

In [11]:
df = pd.read_csv("../data/notes.csv")
df = df[['Field', 'Binary', 'Outflow Source', 'Blue Channels', 'Red Channels', 'Average Angle (Blue)']]
# remove sources with secondaries (deal with those later)
df = df[~df['Binary'].isna()]
# create columns to identify each source in the pair
df['source_a'] = df['Binary'].apply(lambda x: split_source_string(x)[0])
df['source_b'] = df['Binary'].apply(lambda x: split_source_string(x)[1])
# rename columns
df = df[['Field', 'source_a', 'source_b', 'Outflow Source', 'Blue Channels', 'Red Channels', 'Average Angle (Blue)']].rename(columns={'Field': 'field', 'Average Angle (Blue)': 'angle', 'Outflow Source': 'outflow_source', 'Red Channels': 'red_channels', 'Blue Channels': 'blue_channels'})
df

# merge coordinates and distance
new_rows = []
for i, row in df.iterrows():
    key_a = row['field'] + '-' + row['source_a']
    key_b = row['field'] + '-' + row['source_b']
    row_a = source_info.loc[source_info['Source'] == key_a]
    row_b = source_info.loc[source_info['Source'] == key_b]
    row['source_a_ra'] = row_a['RA'].iloc[0]
    row['source_a_dec'] = row_a['Dec'].iloc[0]
    row['source_b_ra'] = row_b['RA'].iloc[0]
    row['source_b_dec'] = row_b['Dec'].iloc[0]
    row['distance'] = row_a['Dis'].iloc[0]
    new_rows.append(row)

master = pd.DataFrame(new_rows)[['field', 'source_a', 'source_a_ra', 'source_a_dec', 'source_b', 'source_b_ra', 'source_b_dec', 'distance', 'outflow_source', 'red_channels', 'blue_channels', 'angle']]

def fix_outflow_source(x):
    if str(x).__contains__('+'):
        return 'both'
    elif len(str(x)) > 1:
        return str(x)[0] + '-' + str(x)[1]
    else:
        return x
    
master['outflow_source'] = master['outflow_source'].apply(lambda x: fix_outflow_source(x))
master.to_csv('../data/outflow_data.csv',index=False)
master

,field,source_a,source_a_ra,source_a_dec,source_b,source_b_ra,source_b_dec,distance,outflow_source,red_channels,blue_channels,angle
0,HOPS-12,B-A,83.787271,-5.931953,B-B,83.787312,-5.931911,388.6,B-A,92-97,103-107,157.1225
2,HOPS-290,A,84.989071,-7.492528,B,84.988871,-7.492419,405.5,A,91-94,99-102,293.5175
3,HOPS-290,A,84.989071,-7.492528,B,84.988871,-7.492419,405.5,B,91-95,NaN,228.8875
4,HOPS-92,A-A,83.826404,-5.009150,A-B,83.826367,-5.009217,392.7,both,NaN,106-112,76.0300
6,HOPS-288,A-A,84.983338,-7.507658,A-B,84.983358,-7.507694,405.5,both,85-91,NaN,56.3700
8,HOPS-203,A,84.095271,-6.768508,B,84.095258,-6.768542,383.5,both,96-100,NaN,143.8100
10,HH270VLA1,A,87.894113,2.946114,B,87.894167,2.946078,430.1,both,94-100,104-109,50.0425
11,HOPS-32,A,83.647725,-5.666481,B,83.647633,-5.666550,390.6,B,95-98,102-106,159.0450
12,HOPS-84,A,83.860667,-5.065311,B,83.860571,-5.065481,392.8,both,98-100,"104, 106-107",82.8150
13,HOPS-168,A,84.078921,-6.756550,B,84.078967,-6.756522,383.3,both,91-98,104-115,159.3500
